In [ ]:
import os, re
from collections import defaultdict
from PIL import Image
from tqdm.notebook import tqdm 

PATCH_RE = re.compile(r'^(?P<prefix>.*?)_(?P<x>\d+)x(?P<y>\d+)\.(?P<ext>jpg|jpeg|png)$', re.I)

def parse_name(fname):
    m = PATCH_RE.match(fname)
    return (m.group('prefix'), int(m.group('x')), int(m.group('y'))) if m else None

def gather_patches(patch_dir):
    groups = defaultdict(list)
    for f in os.listdir(patch_dir):
        info = parse_name(f)
        if info:
            prefix, x, y = info
            groups[prefix].append((os.path.join(patch_dir, f), x, y))
    return groups

In [ ]:
import os, re
from collections import defaultdict
from PIL import Image
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm

PATCH_DIR  = r'../stnet_dataset/cropped_images/'   
SAVE_DIR   = r'./stitched-stnet'                            
SAVE_FLAG  = False                                 
GAP       = 2

groups = gather_patches(PATCH_DIR)
if not groups:
    print('error')

for prefix, items in tqdm(groups.items(), desc='...'):
    # 读第一张拿 patch 尺寸
    first = Image.open(items[0][0])
    pw, ph = first.size
    
    max_x = max(x for _, x, _ in items)
    max_y = max(y for _, _, y in items)
    
    canvas_w = (max_x + 1) * pw + (max_x + 3) * GAP  
    canvas_h = (max_y + 1) * ph + (max_y + 3) * GAP 
    canvas = Image.new('RGB', (canvas_w, canvas_h), color='black') 
    
    for path, x, y in items:
        img = Image.open(path)
        left = GAP + x * (pw + GAP)
        top  = GAP + y * (ph + GAP)
        canvas.paste(img, (left, top))

    if SAVE_FLAG:
        os.makedirs(SAVE_DIR, exist_ok=True)
        canvas.save(os.path.join(SAVE_DIR, f'{prefix}.jpg'), quality=95)
    else:
        canvas.show(title=prefix)
    
    plt.figure(figsize=(6,6))
    plt.imshow(canvas)
    plt.axis('off')
    plt.title(prefix)
    plt.show()
    

In [ ]:
import matplotlib.pyplot as plt
import random

stitched_files = [os.path.join(SAVE_DIR, f) for f in os.listdir(SAVE_DIR) if f.lower().endswith(('.jpg','.png'))]
if stitched_files:
    sample = random.choice(stitched_files)
    img = Image.open(sample)
    plt.figure(figsize=(6,6))
    plt.imshow(img)
    plt.axis('off')
    plt.title(os.path.basename(sample))